In [0]:
from pyspark.sql.functions import trim, col, StringType
import pyspark.sql.functions as F

df = spark.table("dev_project.bronze.erp_px_cat_g1v2")

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

df = df.withColumn(
    "MAINTENANCE",
    F.when(F.upper(F.col("MAINTENANCE")) == "YES", F.lit(True))
    .when(F.upper(F.col("MAINTENANCE")) == "NO", F.lit(False))
    .otherwise(None),
)
name_map = {
    "ID":"id",
    "CAT":"category",
    "SUBCAT":"sub_category",
    "MAINTENANCE":"maintenance"
}

for old_name,new_name in name_map.items():
    df = df.withColumnRenamed(old_name,new_name)

df.display()

df.write.mode("overwrite").saveAsTable("dev_project.silver.erp_px_cat_g1v2")

In [0]:
""